# 🛡️ DriftNet Live: Dual-Head Trajectory Transformer & Real-Time Prefix Monitor

Companion implementation extending **DriftNet** (*Pinjari & Saint-Germain, Northern Arizona University, 2026 — [arXiv:2609.10892](https://arxiv.org/abs/2609.10892)*) with **Section XI Future Work: Real-Time Streaming Prefix Operation & Interactive Triage UI**.

---

### 🚀 How to Use This Notebook (Two Modes)

1. **Mode A — UI Only (Fastest, ~15 seconds):**
   * Run **Section 1** $\rightarrow$ Skip Section 2 $\rightarrow$ Run **Section 3**.
   * Launches the interactive **DriftNet Live Web UI** via a public HTTPS tunnel (`trycloudflare.com`), automatically using pre-saved artifacts (from the repo or your Google Drive).

2. **Mode B — Full GPU Training + Streaming Benchmark + UI (~1 minute on Colab Pro GPU):**
   * Run **Section 1** $\rightarrow$ **Section 2** $\rightarrow$ **Section 3**.
   * Precomputes `all-mpnet-base-v2` embeddings on GPU, trains the `1,845,637`-parameter Dual-Head Transformer from scratch on `data_taskdisjoint`, evaluates both **Full-Trajectory** and **Streaming Prefix** metrics on all 1,722 held-out test trajectories, and launches the Web UI.

---
## Section 1: Environment Setup & Repository Clone *(Always Run First)*

### Cell 1.1 — Clone Repository, Install Dependencies & Verify GPU
**Purpose:** Clones your `AgentDrift` fork (`https://github.com/zeeshanali-k/AgentDrift.git`), navigates into the project directory, installs `sentence-transformers` and `scikit-learn`, and prints the active GPU hardware.

In [ ]:
import os
import subprocess
import torch

REPO_URL = "https://github.com/zeeshanali-k/AgentDrift.git"
REPO_DIR = "/content/AgentDrift"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository already exists. Pulling latest updates...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
print("Current Working Directory:", os.getcwd())

print("Installing Python dependencies (sentence-transformers, scikit-learn)...")
subprocess.run(["pip", "install", "-q", "sentence-transformers", "scikit-learn"], check=True)

if torch.cuda.is_available():
    print(f"✅ GPU Ready: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ℹ️ Running on CPU (sufficient for UI-only mode; enable GPU runtime for fast training).")

### Cell 1.2 *(Optional)* — Mount Google Drive for Persistent Model Checkpoints
**Purpose:** Mounts your Google Drive and links `/content/drive/MyDrive/DriftNet_Artifacts`. If a previously trained `driftnet_best.pt` checkpoint exists on your Drive, it is automatically copied into `driftnet/artifacts/` so you can jump straight to **Section 3 (UI)** without retraining.

In [ ]:
import shutil
from pathlib import Path

USE_GOOGLE_DRIVE = True  # Set to False if you do not want to mount Google Drive
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/DriftNet_Artifacts")
LOCAL_ARTIFACT_DIR = Path("/content/AgentDrift/driftnet/artifacts")
LOCAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        for fname in ["driftnet_best.pt", "eval_results.json"]:
            src = DRIVE_ARTIFACT_DIR / fname
            dst = LOCAL_ARTIFACT_DIR / fname
            if src.exists():
                shutil.copy2(src, dst)
                print(f"✅ Restored {fname} from Google Drive -> {dst}")
        if not (LOCAL_ARTIFACT_DIR / "driftnet_best.pt").exists():
            print("ℹ️ No saved checkpoint on Drive yet. Run Section 2 once to train and save it!")
    except Exception as e:
        print(f"Google Drive mount skipped: {e}")

---
## Section 2: Model Training & Streaming Prefix Benchmark *(Skip if Using UI Only)*

### Cell 2.1 — Precompute & Cache Step Embeddings + World Features
**Purpose:** Serializes all 71,024 steps across the `train` (9,081), `val` (1,733), and `test` (1,722) splits of `data_taskdisjoint` in the paper's fixed format (`TOOL | THOUGHT | ARGS | OBS`), encodes them with frozen `all-mpnet-base-v2` ($\mathbb{R}^{768}$), extracts the 4 identity-free world features ($\mathbb{R}^4$), and caches them to `driftnet/cache/`.

In [ ]:
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer
from driftnet.features import build_or_load_split_cache

device = "cuda" if torch.cuda.is_available() else "cpu"
data_dir = Path("/content/AgentDrift/data_taskdisjoint")
cache_dir = Path("/content/AgentDrift/driftnet/cache")

encoder = SentenceTransformer("all-mpnet-base-v2", device=device)
for split in ["train", "val", "test"]:
    build_or_load_split_cache(split, data_dir / f"{split}.jsonl", cache_dir, encoder=encoder, device=device)

del encoder
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ All split embeddings and world features cached!")

### Cell 2.2 — Train Dual-Head DriftNet (`1,845,637` Params) & Run Full + Streaming Evaluation
**Purpose:** Trains the 3-layer Dual-Head Trajectory Transformer following Algorithm 1 of the paper (class-weighted joint loss $L_{\text{traj}} + L_{\text{step}}$, triangular warmup/decay schedule, early stopping on validation F1). Then evaluates the best checkpoint once on the 1,722 held-out test trajectories in both **Full-Trajectory Mode** and **Real-Time Streaming Prefix Mode**, and backs up the checkpoint to Google Drive.

In [ ]:
import shutil
from pathlib import Path
from driftnet.train_and_eval import main as run_train_and_eval

# Remove existing checkpoint if you want to force a fresh training run
ckpt_file = Path("/content/AgentDrift/driftnet/artifacts/driftnet_best.pt")
if ckpt_file.exists():
    ckpt_file.unlink()

run_train_and_eval()

# Backup trained checkpoint + evaluation metrics to Google Drive if mounted
drive_dir = Path("/content/drive/MyDrive/DriftNet_Artifacts")
local_dir = Path("/content/AgentDrift/driftnet/artifacts")
if drive_dir.exists():
    for fname in ["driftnet_best.pt", "eval_results.json"]:
        if (local_dir / fname).exists():
            shutil.copy2(local_dir / fname, drive_dir / fname)
    print(f"✅ Backed up driftnet_best.pt and eval_results.json to {drive_dir}")

### Cell 2.3 — Inspect Held-Out Test Metrics & Plot Training Dynamics
**Purpose:** Reads `driftnet/artifacts/eval_results.json` and displays a clean summary comparing Full-Trajectory performance against Real-Time Streaming Prefix performance, along with training/validation curves.

In [ ]:
import json
import matplotlib.pyplot as plt

with open("/content/AgentDrift/driftnet/artifacts/eval_results.json") as f:
    res = json.load(f)

tf = res["test_full"]
ts = res["test_streaming"]

print("=====================================================================")
print("  HELD-OUT TASK-DISJOINT TEST RESULTS (1,722 Trajectories)")
print("=====================================================================")
print(f"  Full-Trajectory F1        : {tf['trajectory']['f1']:.4f} (P={tf['trajectory']['precision']:.4f}, R={tf['trajectory']['recall']:.4f})")
print(f"  Exact Entry Match (EM_I)  : {tf['localization']['overall_em_i']:.4f}")
print(f"  Hijacked Span IoU (IoU_H) : {tf['localization']['overall_iou_h']:.4f}")
print(f"  Resisted Attack Flag Rate : {tf['category_flag_rates']['failed_attack']['rate']:.4f} ({tf['category_flag_rates']['failed_attack']['flagged']}/{tf['category_flag_rates']['failed_attack']['total']})")
print("---------------------------------------------------------------------")
print("  REAL-TIME STREAMING PREFIX MONITOR (Section XI Extension)")
print("---------------------------------------------------------------------")
print(f"  Instant 1st-Hijack Catch  : {ts['instant_first_hijack_recall']*100:.1f}% (caught at exact 1st H step)")
print(f"  Online Overall Recall     : {ts['online_overall_recall']*100:.1f}%")
print(f"  Retroactive Entry Match   : {ts['retroactive_entry_point_em_at_halt']*100:.1f}% at moment of halt")
print(f"  Downstream Steps Saved    : {ts['total_downstream_steps_saved']} steps ({ts['mean_downstream_steps_saved']} avg/trace)")
print("=====================================================================")

hist = res.get("train_history", [])
if hist:
    epochs = [h["epoch"] for h in hist]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, [h["train_loss"] for h in hist], label="Train Loss")
    ax1.plot(epochs, [h["val_loss"] for h in hist], label="Val Loss")
    ax1.set_title("Joint Loss (Train vs. Validation)")
    ax1.set_xlabel("Epoch")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(epochs, [h["val_f1"] for h in hist], label="Val Trajectory F1", color="green")
    ax2.plot(epochs, [h["val_em_i"] for h in hist], label="Val Exact Entry (EM_I)", linestyle="--")
    ax2.plot(epochs, [h["val_iou_h"] for h in hist], label="Val Hijacked IoU", linestyle=":")
    ax2.set_title("Validation Detection & Localization")
    ax2.set_xlabel("Epoch")
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## Section 3: Launch Interactive Web UI via Public Tunnel *(Works With or Without Section 2)*

### Cell 3.1 — Start `DriftNet Live` Web Server & Public Cloudflare Tunnel
**Purpose:** Starts the `DriftNet Live` backend server on port `8501` in a background thread and opens a **free, zero-config Cloudflare public HTTPS tunnel (`https://....trycloudflare.com`)** (plus Google Colab's native port proxy). Click the printed URL to open the full interactive dashboard in a new browser tab during your meeting!

In [ ]:
import os
import re
import subprocess
import threading
import time
from pathlib import Path
from http.server import ThreadingHTTPServer
import driftnet.server as dn_server

PORT = 8501
repo_root = Path("/content/AgentDrift")

# 1. Initialize DriftNet App State and start HTTP server in a daemon thread
dn_server.APP_STATE = dn_server.DriftNetAppState(repo_root)

def _start_bg_server():
    httpd = ThreadingHTTPServer(("127.0.0.1", PORT), dn_server.DriftNetRequestHandler)
    httpd.serve_forever()

server_thread = threading.Thread(target=_start_bg_server, daemon=True)
server_thread.start()
time.sleep(1)
print(f"✅ DriftNet Live server running locally on port {PORT}")

# 2. Also provide Google Colab's built-in proxy link
try:
    from google.colab import output
    print("\n🔗 Option A — Colab Native Proxy Link:")
    output.serve_kernel_port_as_window(PORT, path="/")
except Exception:
    pass

# 3. Start Cloudflare Quick Tunnel (no account or token needed)
cf_bin = "/content/cloudflared"
if not os.path.exists(cf_bin):
    print("Downloading cloudflared tunnel binary...")
    subprocess.run(
        ["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin],
        check=True,
    )
    os.chmod(cf_bin, 0o755)

print("Starting public HTTPS tunnel...")
proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n=====================================================================")
    print(f"🚀 PUBLIC UI URL (Click to open): {public_url}")
    print("=====================================================================\n")
else:
    print("⚠️ Could not parse tunnel URL automatically; use the Colab Native Proxy Link above.")

# Keep cell alive while you use the UI (press Stop button in Colab when finished)
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    proc.terminate()
    print("Tunnel stopped.")